### Import Dependencies

In [40]:
import os
from google import genai
from google.genai import types
from qdrant_client import QdrantClient

In [8]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langsmith import Client

### Download an example reference data point from langsmith

In [9]:
from dotenv import load_dotenv
import os

load_dotenv("../../.env")

True

In [10]:
ls_client = Client()

In [12]:
dataset = ls_client.read_dataset(dataset_name="RAG-Eval-Dataset")

In [19]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[20].outputs

{'ground_truth': 'We carry a 512GB Micro SD Card (B0BTDCBWQ6) compatible with cameras, drones, tablets, and smartphones. We also offer the Gulloe 128GB Flash Drive (B0C2JGX2M6) featuring multi-port USB 3.0, lightning, micro USB, and Type-C interfaces for easy data transfers between devices.',
 'reference_context_ids': ['B0BTDCBWQ6', 'B0C2JGX2M6'],
 'reference_descriptions': ['Micro SD Card 512GB Micro Memory SD Cards with A Free SD Card Adapter,TF Card 512GB Memory Card 512GB Fast Speed for Camera/Android Phones [Stable]Memory card is made from high-quality chips, offer reliable performance, ensuring evidence is clearly recorded in full HD without dropped frames. [Durable]Memory cards are temperature waterproof, shock-proof, temperature proof, magnet and x-ray. [High Speed]Memory card up to 15MB/s read speed,write speed is 10MB/s.Based on internal testing; performance may be lower depending on host device, interface, usage conditions and other factors. [Compatibility]Memory card is com

In [20]:
list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[20].inputs

{'question': 'What types of storage expansions do you have, such as flash drives or micro SD cards?'}

In [21]:
reference_input = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[20].inputs
reference_output = list(ls_client.list_examples(dataset_id=dataset.id, limit=50))[20].outputs

### RAG Pipeline

In [23]:
gemini_client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))
qdrant_client = QdrantClient(url="http://localhost:6333")


def get_embeddings(text, task_type="SEMANTIC_SIMILARITY", model="gemini-embedding-001"):
    result = gemini_client.models.embed_content(
        model=model,
        contents=text,
        config=types.EmbedContentConfig(task_type=task_type)
    )
    return result.embeddings[0].values

def retrieve_data(query, k=5):
    query_embedding = get_embeddings(query)
    results = qdrant_client.query_points(
        collection_name="Amazon-items-collection-01",
        query=query_embedding,
        limit=k
    )
    retrieved_context_ids=[]
    retrieved_context=[]
    similarity_scores=[]
    retrieved_context_ratings=[]

    for result in results.points:
        retrieved_context_ids.append(result.payload["parent_asin"])
        retrieved_context.append(result.payload["preprocessed_description"])
        similarity_scores.append(result.score)
        retrieved_context_ratings.append(result.payload["average_rating"])

    return {
        "retrieved_context_ids": retrieved_context_ids,
        "retrieved_context": retrieved_context,
        "similarity_scores": similarity_scores,
        "retrieved_context_ratings": retrieved_context_ratings
    }


def process_context(context):
    formatted_context = ""
    for id, chunk, rating in zip(context["retrieved_context_ids"], context["retrieved_context"], context["retrieved_context_ratings"]):
        formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
    return formatted_context


def build_prompt(query, preprocessed_context):
    prompt = f"""
    You are a helpful assistant that can answer questions about the products in stock.
    You are given a question and a list of context.
    You need to answer the question based on the product descriptions.

    Instructions:
    - Answer the question based on the provided context only.
    - Never use the word "context" in your answer, refer it as the available products.
    - Do not use markdown formatting.

    Context:
    {preprocessed_context}

    Question: 
    {query}
    """
    return prompt

def generate_answer(prompt):
    response = gemini_client.models.generate_content(
    contents=[prompt],
    model="gemini-2.5-flash",
)

    return response.text

def rag_pipeline(query, topk_k=5):
    retrieved_context = retrieve_data(query, topk_k)
    preprocessed_context = process_context(retrieved_context)
    prompt = build_prompt(query, preprocessed_context)
    answer = generate_answer(prompt)

    final_answer = {
        "answer": answer,
        "question": query,
        "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
        "retrieved_context": retrieved_context["retrieved_context"]
    }
    return final_answer


In [24]:
rag_pipeline("can i get a charger?")

{'answer': "Yes, we have chargers available.\n\nWe have a Buedarate Hidden Camera Wall Charger with WiFi Spy Camera, which also functions as an outlet expander with 3 AC outlets and 3 USB PD fast charging ports (3.4A, 20W). It's a 4-in-1 device that can be used for daily charging and home security.\n\nWe also have a 3-pack of iPhone Charger Cables (6FT). These are Apple MFi Certified Lightning cables that support up to 2.4A fast charging and 480 Mbps data transfer. They are nylon braided for durability and compatible with various iPhone models (14, 13, 12, 11 Pro Max, XR, XS, X, 8, 7, 6 Plus SE, iPad).",
 'question': 'can i get a charger?',
 'retrieved_context_ids': ['B0BD5TK2MF',
  'B08DLWKXGL',
  'B0C93Q4CPK',
  'B09V1F24X4',
  'B0B5K9NFHV'],
 'retrieved_context': ['Buedarate Hidden Camera Wall Charger with WiFi Spy Camera Hidden Cameras Outlet HD 1080P Wireless for Home Security Secret Camera with USB Fast Charger 20W PD Charging Port Wall Charger Nanny Cam ✔️【4 IN 1 Function Hidden

### Ragas Metrics

In [25]:
from ragas import SingleTurnSample
from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy

C:\Users\cguti\AppData\Local\Temp\ipykernel_3500\3504057058.py:2: DeprecationWarning: Importing IDBasedContextPrecision from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextPrecision
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\cguti\AppData\Local\Temp\ipykernel_3500\3504057058.py:2: DeprecationWarning: Importing IDBasedContextRecall from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import IDBasedContextRecall
  from ragas.metrics import IDBasedContextPrecision, IDBasedContextRecall, Faithfulness, ResponseRelevancy
C:\Users\cguti\AppData\Local\Temp\ipykernel_3500\3504057058.py:2: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'r

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_google_genai import GoogleGenerativeAIEmbeddings

# 1. Initialize the official LangChain Vertex objects
vertex_chat = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    project=os.getenv("GCP_Project"),
    location="us-central1"
)

vertex_embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-001",           
    project=os.getenv("GCP_Project"),
    location="us-central1"                        
)

# 2. Wrap them for the Ragas runtime
evaluator_llm = LangchainLLMWrapper(vertex_chat)
evaluator_embeddings = LangchainEmbeddingsWrapper(vertex_embeddings)

C:\Users\cguti\AppData\Local\Temp\ipykernel_3500\1224984976.py:20: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_factory; llm = llm_factory('gpt-4o-mini', client=OpenAI(api_key='...'))
  evaluator_llm = LangchainLLMWrapper(vertex_chat)
C:\Users\cguti\AppData\Local\Temp\ipykernel_3500\1224984976.py:21: DeprecationWarning: LangchainEmbeddingsWrapper is deprecated and will be removed in a future version. Use the modern embedding providers instead: embedding_factory('openai', model='text-embedding-3-small', client=openai_client) or from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  evaluator_embeddings = LangchainEmbeddingsWrapper(vertex_embeddings)


In [49]:
reference_input

{'question': 'What types of storage expansions do you have, such as flash drives or micro SD cards?'}

In [50]:
reference_output

{'ground_truth': 'We carry a 512GB Micro SD Card (B0BTDCBWQ6) compatible with cameras, drones, tablets, and smartphones. We also offer the Gulloe 128GB Flash Drive (B0C2JGX2M6) featuring multi-port USB 3.0, lightning, micro USB, and Type-C interfaces for easy data transfers between devices.',
 'reference_context_ids': ['B0BTDCBWQ6', 'B0C2JGX2M6'],
 'reference_descriptions': ['Micro SD Card 512GB Micro Memory SD Cards with A Free SD Card Adapter,TF Card 512GB Memory Card 512GB Fast Speed for Camera/Android Phones [Stable]Memory card is made from high-quality chips, offer reliable performance, ensuring evidence is clearly recorded in full HD without dropped frames. [Durable]Memory cards are temperature waterproof, shock-proof, temperature proof, magnet and x-ray. [High Speed]Memory card up to 15MB/s read speed,write speed is 10MB/s.Based on internal testing; performance may be lower depending on host device, interface, usage conditions and other factors. [Compatibility]Memory card is com

In [51]:
result = rag_pipeline(reference_input["question"])

In [52]:
result

{'answer': 'We have the following storage expansions available:\n\n*   Micro SD Cards: A 512GB Micro SD Card with an adapter, offering fast speed and durability, compatible with cameras, Android phones, computers, laptops, tablets, smartphones, dash cams, camcorders, surveillance, e-readers, and drones.\n*   Flash Drives: A 128GB Flash Drive for iPhone, iPad, Android, and computers, featuring multi-port USB (USB3.0/iPhone/micro USB/Type C adapter), one-click backup, and compatibility with various devices. It also offers high transfer speeds and privacy protection features.',
 'question': 'What types of storage expansions do you have, such as flash drives or micro SD cards?',
 'retrieved_context_ids': ['B0BTDCBWQ6',
  'B0C2JGX2M6',
  'B08DLWKXGL',
  'B0C6PKB5QW',
  'B09YHBXZC8'],
 'retrieved_context': ['Micro SD Card 512GB Micro Memory SD Cards with A Free SD Card Adapter,TF Card 512GB Memory Card 512GB Fast Speed for Camera/Android Phones [Stable]Memory card is made from high-quality c

In [56]:
async def ragas_context_precision_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    scorer = IDBasedContextPrecision()
    return await scorer.single_turn_ascore(sample)

In [57]:
await ragas_context_precision_id_based(result, reference_output)

0.4

In [58]:
async def ragas_context_recall_id_based(run, example):

    sample = SingleTurnSample(
        retrieved_context_ids=run["retrieved_context_ids"],
        reference_context_ids=example["reference_context_ids"]
    )
    scorer = IDBasedContextRecall()
    return await scorer.single_turn_ascore(sample)

In [59]:
await ragas_context_recall_id_based(result, reference_output)

1.0

In [60]:
async def ragas_faithfulness(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = Faithfulness(llm=evaluator_llm)
    return await scorer.single_turn_ascore(sample)

In [61]:
await ragas_faithfulness(result, reference_output)

1.0

In [62]:
async def ragas_relevancy(run, example):

    sample = SingleTurnSample(
        user_input=run["question"],
        response=run["answer"],
        retrieved_contexts=run["retrieved_context"],
    )
    scorer = ResponseRelevancy(llm=evaluator_llm, embeddings=evaluator_embeddings)
    return await scorer.single_turn_ascore(sample)

In [63]:
await ragas_relevancy(result, reference_output)

np.float64(0.7772117464291606)